In [1]:
import cv2

import os
os.environ["QT_QPA_PLATFORM"] = "xcb"

import sys
from pathlib import Path
ROOT = Path(Path.cwd().parent)
sys.path.append(str(ROOT))

from src.computer_vision.detector import VehicleDetector
from src.computer_vision.tracker_ultralytics import UltralyticsTracker
from src.utils.roi_selector import ROI
from src.computer_vision.roi_loader import load_rois
from matplotlib import pyplot as plt

/home/Abo/Projects/Traffic Analysis


In [2]:
CONF_THRESH = 0.5
CONFIG_PATH = ROOT / "src" / "config.json"

In [3]:
video_path1 = Path.joinpath(ROOT, "data", "dataset", "normanniles1", "normanniles1_2025-10-21-13-01-45.mp4")
video_path2 = Path.joinpath(ROOT, "data", "dataset", "normanniles2", "normanniles2_2025-10-23-16-01-45.mp4")
video_path3 = Path.joinpath(ROOT, "data", "dataset", "normanniles3", "normanniles3_2025-10-21-16-54-45.mp4")
video_path4 = Path.joinpath(ROOT, "data", "dataset", "normanniles4", "normanniles4_2025-10-21-15-24-45.mp4")

print(Path.exists(video_path1))

True


In [ ]:
cap = cv2.VideoCapture(str(video_path4))
detector = VehicleDetector(conf=CONF_THRESH)
udetector = UltralyticsTracker(conf=CONF_THRESH)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # detections = detector.detect(frame)
    detections = udetector.track(frame)

    for det in detections:
        x1, y1, x2, y2 = det["bbox"]
        conf = det["conf"]
        track_id = det["track_id"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"{conf:.2f}",
            (x1, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1
        )
        cv2.putText(
            frame,
            f"{track_id}",
            (x1 + 20, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 0, 0),
            1
        )
    
    rois = load_rois(str(CONFIG_PATH))
    roi_entry = rois["camera_4"]["entry"]
    roi_exit = rois["camera_4"]["exit"]
    frame = roi_entry.draw(frame=frame)
    frame = roi_exit.draw(frame=frame, color=(0, 0, 255))

    cv2.imshow("Video", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()